# Mod3D Usage Examples

Each cell demonstrates a feature of the Mod3D Python bindings for Open Cascade Technology, followed by an interactive 3D rendering.

In [1]:
# Shared renderer setup
from mod3d import ShapeRenderer

def render(*shapes, **kwargs):
    """Quick helper to render one or more shapes."""
    r = ShapeRenderer()
    r.width = 800
    r.heigh = 600
    for s in shapes:
        r.add_shape(s)
    return r.render(background='lightgray', **kwargs)

## Geometric Primitives

In [2]:
from mod3d import gp

# Points, vectors, directions
p = gp.Pnt(1.0, 2.0, 3.0)
v = gp.Vec(1.0, 0.0, 0.0)
d = gp.Dir(0.0, 0.0, 1.0)

# Transformations
trsf = gp.Trsf()
trsf.set_translation(gp.Vec(5.0, 0.0, 0.0))

# Direct transformation methods
ax = gp.Ax1(gp.Pnt(0, 0, 0), gp.Dir(0, 0, 1))
p2 = gp.Pnt(1, 2, 3).translated(gp.Vec(10, 0, 0))
ax2 = ax.rotated(gp.Ax1(gp.Pnt(0, 0, 0), gp.Dir(0, 0, 1)), 1.57)

print(f"Original point: {p}")
print(f"Translated point: {p2}")

Original point: gp_Pnt(1.000000, 2.000000, 3.000000)
Translated point: gp_Pnt(11.000000, 2.000000, 3.000000)


## Creating Shapes — Primitives

In [3]:
from mod3d import BRepBuilderAPI, gp

box = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
render(box)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

In [4]:
sphere = BRepBuilderAPI.MakeSphere(5.0).shape()
render(sphere)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

In [5]:
cylinder = BRepBuilderAPI.MakeCylinder(2.0, 15.0).shape()
render(cylinder)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

In [6]:
cone = BRepBuilderAPI.MakeCone(10.0, 5.0, 20.0).shape()
render(cone)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Creating Shapes — Edges, Wires, and Faces

In [7]:
from mod3d import BRepBuilderAPI, gp

p1, p2 = gp.Pnt(0, 0, 0), gp.Pnt(10, 0, 0)
edge = BRepBuilderAPI.MakeEdge(p1, p2).edge()
wire = BRepBuilderAPI.MakeWire(edge).wire()
render(wire)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## B-Spline Curves

In [8]:
from mod3d import gp, Geom, BRepBuilderAPI

# B-Spline from gp.Pnt list
curve = Geom.BSplineCurve(
    poles=[gp.Pnt(0, 0, 0), gp.Pnt(1, 1, 0), gp.Pnt(2, 1, 0), gp.Pnt(3, 0, 0)],
    knots=[0.0, 1.0, 2.0],
    multiplicities=[3, 1, 3],
    degree=2,
)
edge = BRepBuilderAPI.MakeEdge(curve).edge()
render(edge)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

In [9]:
import numpy as np
np.array([[0, 0, 0], [10, 0, 0], [10, 10, 0], [10, 10, 10]]).shape

(4, 3)

In [10]:
import numpy as np
from mod3d import Geom, BRepBuilderAPI

# B-Spline from numpy arrays with weights
curve_np = Geom.BSplineCurve(
    poles=np.array([[0, 0, 0], [10, 0, 0], [10, 10, 0], [10, 10, 10]]),
    weights=[1.0, 1.2, 0.8, 0.5],
    knots=[0, 1],
    multiplicities=[4, 4],
    degree=3,
)
edge_np = BRepBuilderAPI.MakeEdge(curve_np).edge()
render(edge_np)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Boolean Operations

In [11]:
import mod3d

sphere = mod3d.BRepBuilderAPI.MakeSphere(5.0).shape()
box = mod3d.BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()

# Move box to overlap
trsf = mod3d.gp.Trsf()
trsf.set_translation(mod3d.gp.Vec(-10.0, -10.0, -10.0))
box = box.moved(trsf)

# Fuse (union)
fuse = mod3d.BooleanOp.Fuse(box, sphere)
fuse_shape = fuse.shape()
render(fuse_shape)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

In [12]:
# Common (intersection)
common = mod3d.BooleanOp.Common(sphere, box)
render(common.shape())

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

In [13]:
# Cut (subtraction)
cut = mod3d.BooleanOp.Cut(box, sphere)
render(cut.shape())

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

In [14]:
section = mod3d.BooleanOp.Section(box, sphere).shape()
render(section)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Filleting

In [15]:
from mod3d import BRepFillet, BRepBuilderAPI, BooleanOp, gp

# Create two overlapping shapes and fuse them
s = BRepBuilderAPI.MakeSphere(5.0).shape()
b = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
t = gp.Trsf()
t.set_translation(gp.Vec(-10.0, -10.0, -10.0))
b = b.moved(t)
fuse = BooleanOp.Fuse(b, s)
result = fuse.shape()

# 3D fillet on section edges
fillet_maker = BRepFillet.MakeFillet(result)
for edge in fuse.section_edges():
    fillet_maker.add(1.0, edge)
filleted = fillet_maker.shape()
render(filleted)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Curve Interpolation

In [16]:
from mod3d import gp, GeomAPI, BRepBuilderAPI

points = [gp.Pnt(0, 0, 0), gp.Pnt(1, 1, 0), gp.Pnt(2, 0, 0)]
interp = GeomAPI.Interpolate(points)
interp.perform()
curve = interp.curve
edge = BRepBuilderAPI.MakeEdge(curve).edge()
render(edge)

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Distance Computation

In [17]:
from mod3d import BRepExtrema, BRepBuilderAPI, gp

box1 = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
box2 = BRepBuilderAPI.MakeBox(gp.Pnt(20.0, 0.0, 0.0), 10.0, 10.0, 10.0).shape()

dist = BRepExtrema.DistShapeShape(box1, box2)
print(f"Distance between boxes: {dist.value}")

render(box1, box2)

Distance between boxes: 10.0


Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Global Properties

In [18]:
from mod3d import BRepGProp, BRepBuilderAPI

box = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
props = BRepGProp.BRepGProp.linear_properties(box)
print(f"Centre of mass: {props.centre_of_mass}")

render(box)

Centre of mass: gp_Pnt(5.000000, 5.000000, 5.000000)


Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Tessellation

In [19]:
from mod3d import Render, BRepBuilderAPI

box = BRepBuilderAPI.MakeBox(10.0, 20.0, 30.0).shape()
faces, edges = Render.extract_tessellation(box, linear_deflection=0.1)

for triangles, vertices, normals, uvs in faces:
    print(f"{vertices.shape[0]} vertices, {triangles.shape[0]} triangles")

render(box)

4 vertices, 2 triangles
4 vertices, 2 triangles
4 vertices, 2 triangles
4 vertices, 2 triangles
4 vertices, 2 triangles
4 vertices, 2 triangles


Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…

## Interactive Rendering — Custom Options

In [23]:
from mod3d import ShapeRenderer, BRepBuilderAPI, gp

# Build a scene with multiple primitives
box = BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()
sphere = BRepBuilderAPI.MakeSphere(gp.Pnt(15, 5, 5), 4.0).shape()
cyl = BRepBuilderAPI.MakeCylinder(2.0, 12.0).shape()

renderer = ShapeRenderer()
renderer.angle_deflection = 5
renderer.linear_deflection = 0.01
renderer.width =800
renderer.height=600
renderer.add_shape(box, {'surface_color': '#ff6600'})
renderer.add_shape(sphere, {'surface_color': '#0066ff'})
renderer.add_shape(cyl, {'surface_color': '#00cc44'})
renderer.render(background='lightgray')

Box(children=(Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, far=1000.0, fov=45.0, position=(0.0…